In [1]:
from ollama import chat
from ollama import ChatResponse
from ollama import Client
import pandas as pd
import json
import time
import random
import numpy as np
import os
from tqdm import tqdm
import gc
import re
import warnings
warnings.filterwarnings("ignore")

### Setup RAG

- Define output schema

In [2]:
from typing import List
from pydantic import BaseModel


class prediction(BaseModel):
    T: str
    N: str
    stage: str
    reason: str

json_schema = prediction.model_json_schema()
json_schema


{'properties': {'T': {'title': 'T', 'type': 'string'},
  'N': {'title': 'N', 'type': 'string'},
  'stage': {'title': 'Stage', 'type': 'string'},
  'reason': {'title': 'Reason', 'type': 'string'}},
 'required': ['T', 'N', 'stage', 'reason'],
 'title': 'prediction',
 'type': 'object'}

- create document converter

In [3]:
# Create document store and preprocessing pipeline with metadata enrichment
from haystack import Pipeline, component
from haystack.components.writers import DocumentWriter
from haystack.components.converters import MarkdownToDocument, PyPDFToDocument, TextFileToDocument
from haystack.components.preprocessors import DocumentSplitter, DocumentCleaner
from haystack.components.routers import FileTypeRouter
from haystack.components.joiners import DocumentJoiner
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import Document

from typing import List, Dict, Any
from pathlib import Path
from datetime import datetime
import mimetypes

# Embedding model and document store
# Using BiomedNLP-PubMedBERT for better medical terminology understanding
embedding_model = "all-mpnet-base-v2"  # High-quality model for scientific/medical papers
document_store = InMemoryDocumentStore()

# File routing and converters
file_type_router = FileTypeRouter(mime_types=["text/plain", "application/pdf", "text/markdown"])
text_file_converter = TextFileToDocument()
markdown_converter = MarkdownToDocument()
pdf_converter = PyPDFToDocument()

# Preprocessing components
document_joiner = DocumentJoiner()
document_cleaner = DocumentCleaner()
document_splitter = DocumentSplitter(split_by="word", split_length=150, split_overlap=30)

document_embedder = SentenceTransformersDocumentEmbedder(model=embedding_model)
document_writer = DocumentWriter(document_store)

# Custom component to enrich metadata
@component
class MetadataEnricher:
    @component.output_types(documents=List[Document])
    def run(self, documents: List[Document]) -> Dict[str, Any]:
        for doc in documents:
            meta = getattr(doc, "meta", None)
            if meta is None:
                doc.meta = {}
                meta = doc.meta

            # Source information
            file_path = meta.get("file_path")
            if file_path:
                meta["source_file"] = str(file_path)
                meta["file_name"] = Path(file_path).name
                #cancer type=file name onyly
                meta['cancer_type'] = Path(file_path).name.split('.')[0]
                meta["source_dir"] = str(Path(file_path).parent)
                mime = mimetypes.guess_type(str(file_path))[0]
                if mime:
                    meta["mime_type"] = mime

            # Processing info
            meta["processed_at"] = datetime.now().isoformat()
            meta["ingestion_pipeline"] = "preprocess_v1"

            # Content info
            if getattr(doc, "content", None):
                meta["content_length"] = len(doc.content)
                if "content_preview" not in meta:
                    meta["content_preview"] = doc.content[:120]
        return {"documents": documents}

metadata_enricher = MetadataEnricher()

# Build preprocessing pipeline with metadata enrichment
preprocessing_pipeline = Pipeline()
preprocessing_pipeline.add_component(instance=file_type_router, name="file_type_router")
preprocessing_pipeline.add_component(instance=text_file_converter, name="text_file_converter")
preprocessing_pipeline.add_component(instance=markdown_converter, name="markdown_converter")
preprocessing_pipeline.add_component(instance=pdf_converter, name="pypdf_converter")
preprocessing_pipeline.add_component(instance=document_joiner, name="document_joiner")
preprocessing_pipeline.add_component(instance=document_cleaner, name="document_cleaner")
preprocessing_pipeline.add_component(instance=document_splitter, name="document_splitter")
preprocessing_pipeline.add_component(instance=metadata_enricher, name="metadata_enricher")
preprocessing_pipeline.add_component(instance=document_embedder, name="document_embedder")
preprocessing_pipeline.add_component(instance=document_writer, name="document_writer")

# Connect the components
preprocessing_pipeline.connect("file_type_router.text/plain", "text_file_converter.sources")
preprocessing_pipeline.connect("file_type_router.application/pdf", "pypdf_converter.sources")
preprocessing_pipeline.connect("file_type_router.text/markdown", "markdown_converter.sources")
preprocessing_pipeline.connect("text_file_converter", "document_joiner")
preprocessing_pipeline.connect("pypdf_converter", "document_joiner")
preprocessing_pipeline.connect("markdown_converter", "document_joiner")
preprocessing_pipeline.connect("document_joiner", "document_cleaner")
preprocessing_pipeline.connect("document_cleaner", "document_splitter")
preprocessing_pipeline.connect("document_splitter", "metadata_enricher")
preprocessing_pipeline.connect("metadata_enricher", "document_embedder")
preprocessing_pipeline.connect("document_embedder", "document_writer")

print("Preprocessing pipeline with metadata enrichment is ready.")

Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1030 10:26:30.137000 24760 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Preprocessing pipeline with metadata enrichment is ready.


In [4]:
# Ingest all files from the folder into the document store
# Collect files (recursively) under the folder
folder = r"E:\head_neck_staging\knowledge_document"

sources = [p for p in Path(folder).rglob("*") if p.is_file()]
print(f"Found {len(sources)} files to ingest from: {folder}")

# Run the pipeline
if sources:
    preprocessing_pipeline.run({"file_type_router": {"sources": sources}})
    print("Ingestion completed.")
else:
    print("No files found to ingest.")

Found 7 files to ingest from: E:\head_neck_staging\knowledge_document
Ingestion completed.


In [5]:
from docx import Document as DocxDocument

def docx_to_plain_documents(paths):
    docs=[]
    for p in paths:
        try:
            d=DocxDocument(str(p))
            full_text=[]
            for para in d.paragraphs:
                if para.text.strip():
                    full_text.append(para.text)
            content="\n".join(full_text)
            docs.append(Document(content=content, meta={"file_path": str(p), "original_ext": ".docx"}))
        except Exception as e:
            print(f"Failed to read {p}: {e}")
    return docs

# Use manual conversion then continue with splitter->enricher->embedder->writer
manual_docx_sources=[p for p in Path(folder).rglob("*.docx")]
manual_docs=docx_to_plain_documents(manual_docx_sources)
print(f"Manually converted {len(manual_docs)} docx files.")

# Run remaining pipeline components manually since we bypassed router/converters
cleaned = document_cleaner.run(documents=manual_docs)["documents"]
split_docs = document_splitter.run(documents=cleaned)["documents"]
enriched = metadata_enricher.run(documents=split_docs)["documents"]
embedded = document_embedder.run(documents=enriched)["documents"]
_ = document_writer.run(documents=embedded)
print(f"Document store now has {len(document_store.filter_documents())} documents after manual DOCX ingestion.")

Manually converted 7 docx files.


Batches: 100%|██████████| 2/2 [00:00<00:00,  6.80it/s]

Document store now has 34 documents after manual DOCX ingestion.


In [6]:
# Verify stored documents and metadata
all_docs = document_store.filter_documents()
print(f"Total documents in store: {len(all_docs)}")

for i, doc in enumerate(all_docs[:3]):
    print(f"\n--- Document {i+1} ---")
    print(f"Content preview: {doc.content[:120] if doc.content else ''}")
    print(f"Metadata: {doc.meta}")
    print(f"Embedding present: {doc.embedding is not None}")

Total documents in store: 34

--- Document 1 ---
Content preview: American Joint Committee on Cancer (AJCC) TNM Staging System for the Larynx (8th ed., 2017)
(Nonepithelial tumors such a
Metadata: {'file_path': 'E:\\head_neck_staging\\knowledge_document\\ca_larynx.docx', 'original_ext': '.docx', 'source_id': '2e67a28c1b543c1d7b92745c9d25e8dfb35cc4af8d4c0fbe7de405fc03f73439', 'page_number': 1, 'split_id': 0, 'split_idx_start': 0, '_split_overlap': [{'doc_id': '89011b3429af603d5aca8ca9b3d0f2fd508f5d36c1cb63863d1348a51c08b8b5', 'range': (0, 218)}], 'source_file': 'E:\\head_neck_staging\\knowledge_document\\ca_larynx.docx', 'file_name': 'ca_larynx.docx', 'cancer_type': 'ca_larynx', 'source_dir': 'E:\\head_neck_staging\\knowledge_document', 'mime_type': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'processed_at': '2026-10-30T10:26:36.508959', 'ingestion_pipeline': 'preprocess_v1', 'content_length': 1113, 'content_preview': 'American Joint Committee on Cancer (A

### Setup LLM

- create LLM client

In [18]:
from haystack_integrations.components.generators.ollama import OllamaGenerator
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack.dataclasses import ChatMessage

model_name = "gpt-oss:20b"
seed_list=[15,2655,454233,9672678,132323989]
seed_num = seed_list[4]
print(f"Using seed number: {seed_num}")
generator = OllamaChatGenerator(model=model_name, timeout=200, url='http://localhost:11434',
                             generation_kwargs={
                              "top_k": 5,
                              "temperature": 0.1,
                              "seed": seed_num,
                              "format": json_schema,
                              #"Content-Type":'application/json', 
                              "timeout": 200
                              })


Using seed number: 132323989


In [20]:
from haystack.dataclasses import ChatMessage
def create_rag_template():
    system_message = ChatMessage.from_system(
        """
        You are a medical AI assistant with expertise in head neck cancer staging and management. Your task is to classify cancer stage that require the pathologic repoart and/or radiology report.
        T staging is based on the size and extent of the primary tumor. N staging is based on the involvement of regional lymph nodes.
        Overall stage grouping combines T and N classifications to determine the overall stage of the cancer.
        Follow this systematic approach:
        Step 1. Determine primary cancer site
        Step 2. Primary tumor assessment from pathologic report (if available)(1st priority) and radiologic investigation  (if available).
        Step 3. T stage assessment according to AJCC staging criteria depend on primary site and findings from step 2.
        Step 4. Neck lymph node assessment from pathologic report (if available)(1st priority) and radiologic investigation  (if available).
        Step 5. N stage assessment according to AJCC staging criteria depend on primary site and findings from step 4.
        Step 6. AJCC staging classification (8th edition) for head neck cancer.
        *Note :  -DOI is depth of invasion and not tumor thickness.
                 -ENE is extranodal extension.
                 -The patholoic lymph node in radiology report is size more than 1 cm and show necrosis or irregular margin to consider as pathologic lymph node. 
                 - Pathologic node location is affeected the AJCC N staging base on cancer type.
        If you are not sure about the stage, try your best to answer. Answer in JSON format as example below :
        * Note:and do not use (```json) in the response
        {{schema}}
        This is an example response:
        {"T": "T2",
        "N": "N1",
        "stage": "Stage III",
        "reason": "Based on the primary tumor size of 3 cm and involvement of a single ipsilateral lymph node less than 3 cm without ENE, the T2 and N1 classifications lead to an overall Stage III classification according to AJCC 8th edition criteria."
         """
    )
    
    user_message = ChatMessage.from_user(
        """
        "Context information:\n"
        "{% for document in documents %}"
        "{{ document.content }}\n"
        "{% endfor %}"
        "\n"
        From the provided patient information . please classify patient stage according to AJCC staging. Answer T, N and overall stage (all case are M0).
        The AJCC staging document of each cancer is specified in the document meta 'cancer_type'. 
         \n
         This is the document(Input) that you have to classify stage
         \n
      {{question}}
   """
    )
    return [system_message, user_message]

template = create_rag_template()
template

[ChatMessage(_role=<ChatRole.SYSTEM: 'system'>, _content=[TextContent(text='\n        You are a medical AI assistant with expertise in head neck cancer staging and management. Your task is to classify cancer stage that require the pathologic repoart and/or radiology report.\n        T staging is based on the size and extent of the primary tumor. N staging is based on the involvement of regional lymph nodes.\n        Overall stage grouping combines T and N classifications to determine the overall stage of the cancer.\n        Follow this systematic approach:\n        Step 1. Determine primary cancer site\n        Step 2. Primary tumor assessment from pathologic report (if available)(1st priority) and radiologic investigation  (if available).\n        Step 3. T stage assessment according to AJCC staging criteria depend on primary site and findings from step 2.\n        Step 4. Neck lymph node assessment from pathologic report (if available)(1st priority) and radiologic investigation  (if

In [21]:
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
#from haystack.components.generators.chat import HuggingFaceAPIChatGenerator

# Build RAG pipeline with optimized retrieval
pipe = Pipeline()
pipe.add_component("embedder", SentenceTransformersTextEmbedder(model=embedding_model))
# Increase top_k to retrieve more documents for better context
pipe.add_component("retriever", InMemoryEmbeddingRetriever(document_store=document_store, top_k=5))
pipe.add_component("chat_prompt_builder", ChatPromptBuilder(template=template))
pipe.add_component(
    "llm",
    generator,
)

pipe.connect("embedder.embedding", "retriever.query_embedding")
pipe.connect("retriever.documents", "chat_prompt_builder.documents")
pipe.connect("chat_prompt_builder.prompt", "llm.messages")

ChatPromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


🚅 Components
  - embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - chat_prompt_builder: ChatPromptBuilder
  - llm: OllamaChatGenerator
🛤️ Connections
  - embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> chat_prompt_builder.documents (list[Document])
  - chat_prompt_builder.prompt -> llm.messages (list[ChatMessage])

- Load data

In [32]:
#load data
df=pd.read_excel(r"E:\head_neck_staging\HN_cancer_sample_data.xlsx")
print(f"Initial data shape: {df.shape}")

df

Initial data shape: (5, 7)


,ID,diagnosis,T_stage,N_stage,Overall_stage,Pathology_result,Radiology_result
0,1,"Lymph nodes of head, face and neck malignant n...",0,2,4,"Review slide\nI. Left cervical lymph node, lev...",NECK\n\nNeck: Mild rightward deviation of nasa...
1,2,Malignant neoplasm of base of tongue,2,2,3,NaN,1. Enhancing ulcerative mass at posterior 1/3 ...
2,3,"Larynx malignant neoplasm, unspecified",3,0,3,S/P DL c Esophagoscope c TLG 07/12/66 ต้นสังกั...,The study reveals markedly increase in size of...
3,4,"Oropharynx malignant neoplasm, unspecified",1,0,1,NaN,The study reveals a 1.3x1.2x1.0-cm well-define...
4,5,"Tongue malignant neoplasm, unspecified",3,2,4,NaN,Finding ; - A large well defined diffuse enhan...


### Run 1 sample

In [33]:
data=df.iloc[0]
data

ID                                                                  1
diagnosis           Lymph nodes of head, face and neck malignant n...
T_stage                                                             0
N_stage                                                             2
Overall_stage                                                       4
Pathology_result    Review slide\nI. Left cervical lymph node, lev...
Radiology_result    NECK\n\nNeck: Mild rightward deviation of nasa...
Name: 0, dtype: object

In [34]:
#create question from ecog, ChildPugh, DOctorNote using f
question = f"""This patient had cancer of {data['diagnosis']} . He/She had radiology report as {data['Radiology_result']} and pathologic result was {data['Pathology_result']} """
question

'This patient had cancer of Lymph nodes of head, face and neck malignant neoplasm (metastatic Mucoepidermoid carcinoma at neck node with unknown primary) . He/She had radiology report as NECK\n\nNeck: Mild rightward deviation of nasal septum. Unremarkable nasopharynx, oropharynx\nand larynx. Normal appearance of submandibular and parotid glands.\n\nThyroid: A 2.5x1.8-cm solid enhancing nodule with partial rim and internal calcification at\nleft thyroid lobe.\n\nLymph nodes: Multiple enlarged lymph nodes with internal calcifications at left cervical\nlevel IIb, Ill, Va, measured up to 3.0x2.3 cm.\n\n_ Vascular structures: Unremarkable.\n\nVisualized brain: No intracranial hemorrhage or large territorial infarction. No gross\nabnormal enhancing lesion.\n\nOrbits, paranasal sinuses and skull base: Mucosal thickening at right ethmoid sinus.\nBony structures: Decreased cervical lordosis. Mild degenerative change of cervical spine.\nNo suspicious osteolytic or osteoblastic lesion. and pathol

In [35]:
response=pipe.run({"embedder": {"text": question}, "chat_prompt_builder": {"question": question,"schema": json_schema}})
response

Batches: 100%|██████████| 1/1 [00:00<00:00, 28.68it/s]


{'llm': {'replies': [ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[ReasoningContent(reasoning_text='We need to classify stage according to AJCC staging for "cancer of Lymph nodes of head, face and neck malignant neoplasm (metastatic Mucoepidermoid carcinoma at neck node with unknown primary)". The meta \'cancer_type\' presumably indicates the staging system to use. The input says "This patient had cancer of Lymph nodes of head, face and neck malignant neoplasm (metastatic Mucoepidermoid carcinoma at neck node with unknown primary)". So it\'s an unknown primary with metastatic cervical lymph nodes. The staging system for "Cervical Lymph Nodes and Unknown Primary Tumors of the Head and Neck (8th ed., 2017)" is provided in the context. That system is for squamous cell carcinoma and salivary gland carcinoma of all head and neck sites except HPV-related oropharynx cancer, nasopharynx cancer, melanoma, thyroid carcinoma, and sarcoma. It includes staging for occult primary tu

In [36]:
def extract_llm_response(response):
    c=response['llm']['replies'][0].text
    match = re.search(r'\{.*\}', c, re.DOTALL)
    if match:
     json_str = match.group(0)
     try:
        data = json.loads(json_str)
        return data
     except json.JSONDecodeError as e:
        print(f"Failed to parse: {e}")

In [37]:
json_resp=extract_llm_response(response)
json_resp

{'T': 'T0',
 'N': 'N1',
 'stage': 'Stage II',
 'reason': 'The patient has an unknown primary head and neck cancer with metastatic mucoepidermoid carcinoma in cervical lymph nodes. The primary tumor cannot be assessed, so T0. Pathologic examination shows 2 metastatic lymph nodes (4.3 and 4.7\u202fcm) with no extranodal extension, fitting pN1 (≤4 nodes). According to AJCC 8th edition staging for unknown primary, T0\u202fN1\u202fM0 corresponds to Stage\u202fII.'}

### run 1 seed

In [44]:
df_sample=df.copy()
df_sample['seed'] = seed_num
df_sample['response']=''
df_sample['T_prediction'] = ''
df_sample['N_prediction']=''
df_sample['Stage_prediction']=''
df_sample['reason']=''  
df_sample['Time']=''
success_count=0
failure_count=0
df_sample

,ID,diagnosis,T_stage,N_stage,Overall_stage,Pathology_result,Radiology_result,seed,response,T_prediction,N_prediction,Stage_prediction,reason,Time
0,1,"Lymph nodes of head, face and neck malignant n...",0,2,4,"Review slide\nI. Left cervical lymph node, lev...",NECK\n\nNeck: Mild rightward deviation of nasa...,132323989,,,,,,
1,2,Malignant neoplasm of base of tongue,2,2,3,NaN,1. Enhancing ulcerative mass at posterior 1/3 ...,132323989,,,,,,
2,3,"Larynx malignant neoplasm, unspecified",3,0,3,S/P DL c Esophagoscope c TLG 07/12/66 ต้นสังกั...,The study reveals markedly increase in size of...,132323989,,,,,,
3,4,"Oropharynx malignant neoplasm, unspecified",1,0,1,NaN,The study reveals a 1.3x1.2x1.0-cm well-define...,132323989,,,,,,
4,5,"Tongue malignant neoplasm, unspecified",3,2,4,NaN,Finding ; - A large well defined diffuse enhan...,132323989,,,,,,


In [39]:
def extract_llm_response(response):
    c=response['llm']['replies'][0].text
    match = re.search(r'\{.*\}', c, re.DOTALL)
    if match:
     json_str = match.group(0)
     try:
        data = json.loads(json_str)
        return data
     except json.JSONDecodeError as e:
        print(f"Failed to parse: {e}")

In [40]:
def compute_response_time(response):
 meta = response['llm']['replies'][0].meta
 time=meta['total_duration']/1_000_000_000
 return    time

In [45]:
#model_name = "qwen3:32b" 
#df_sample=df.copy()
#start_time = time.time()
success_count=0
failure_count=0
for i in tqdm(range(len(df_sample))):
    
    dx=df_sample['diagnosis'].iloc[i]
    radio=df_sample['Radiology_result'].iloc[i]
    patho=df_sample['Pathology_result'].iloc[i]


    question = f"""This patient had cancer of {dx} . He/She had radiology report as {radio} and pathologic result was {patho} """
    try:
     response=pipe.run({"embedder": {"text": question}, "chat_prompt_builder": {"question": question,"schema": json_schema}})
     time_taken=compute_response_time(response)
     try:
      result_json=extract_llm_response(response)
      df_sample['T_prediction'].iloc[i] = result_json['T']
      df_sample['N_prediction'].iloc[i] = result_json['N']
      df_sample['Stage_prediction'].iloc[i] = result_json['stage']
      df_sample['reason'].iloc[i] = result_json['reason'] 
      df_sample['Time'].iloc[i]=time_taken
      print("✓ LLM extraction successful")
      success_count+=1
      print(f"T: {result_json['T']}, N: {result_json['N']}, Stage: {result_json['stage']}")
     
     except:
      df_sample['response'].iloc[i] = response['llm']['replies'][0].text
      df_sample['Time'].iloc[i]=time_taken
      print("✗ LLM extraction failed")
      failure_count+=1
    except:
     df_sample['response'].iloc[i] = "RAG pipeline failed"
     failure_count+=1
     print("✗ RAG pipeline failed")
print(f"Total successful extractions: {success_count}")
print(f"Total failed extractions: {failure_count}")

 20%|██        | 1/5 [00:07<00:28,  7.09s/it]

Failed to parse: Unterminated string starting at: line 1 column 50 (char 49)
✗ LLM extraction failed


 40%|████      | 2/5 [00:09<00:13,  4.50s/it]

✓ LLM extraction successful
T: T2, N: N1, Stage: Stage III


 60%|██████    | 3/5 [00:12<00:07,  3.88s/it]

✓ LLM extraction successful
T: T3, N: N0, Stage: Stage III


 80%|████████  | 4/5 [00:17<00:04,  4.08s/it]

✓ LLM extraction successful
T: T1, N: N2c, Stage: Stage IVA


100%|██████████| 5/5 [00:23<00:00,  4.75s/it]

✓ LLM extraction successful
T: T2, N: N2b, Stage: Stage IVa
Total successful extractions: 4
Total failed extractions: 1


### Check result

In [46]:
df_sample

,ID,diagnosis,T_stage,N_stage,Overall_stage,Pathology_result,Radiology_result,seed,response,T_prediction,N_prediction,Stage_prediction,reason,Time
0,1,"Lymph nodes of head, face and neck malignant n...",0,2,4,"Review slide\nI. Left cervical lymph node, lev...",NECK\n\nNeck: Mild rightward deviation of nasa...,132323989,"{""T"":""TX"",""N"":""N2b"",""stage"":""Stage III"",""reaso...",,,,,6.980109
1,2,Malignant neoplasm of base of tongue,2,2,3,NaN,1. Enhancing ulcerative mass at posterior 1/3 ...,132323989,,T2,N1,Stage III,The primary tumor measures 3.9 cm in greatest ...,2.650313
2,3,"Larynx malignant neoplasm, unspecified",3,0,3,S/P DL c Esophagoscope c TLG 07/12/66 ต้นสังกั...,The study reveals markedly increase in size of...,132323989,,T3,N0,Stage III,The pathology report indicates the tumor invad...,3.125667
3,4,"Oropharynx malignant neoplasm, unspecified",1,0,1,NaN,The study reveals a 1.3x1.2x1.0-cm well-define...,132323989,,T1,N2c,Stage IVA,The radiology report shows a 1.3‑cm lesion on ...,4.360019
4,5,"Tongue malignant neoplasm, unspecified",3,2,4,NaN,Finding ; - A large well defined diffuse enhan...,132323989,,T2,N2b,Stage IVa,The right tongue mass measures approximately 3...,6.417494


In [47]:
def llm_extract_staging(raw_response: str, generator_instance) -> dict:
    """
    Use LLM to extract T, N, and stage from raw LLM response.
    
    Args:
        raw_response: Raw text response from the staging LLM
        generator_instance: Your OllamaChatGenerator instance
    
    Returns:
        dict: Extracted {T, N, stage, reason} in JSON format
    """
    from haystack.dataclasses import ChatMessage
    import json
    import re
    
    extraction_prompt = f"""You are a JSON extraction assistant. Extract the cancer staging information from the following text and return ONLY a valid JSON object with these exact keys: T, N, stage, reason.

Text to extract from:
{raw_response}

Return your answer as a JSON object with this exact format:
{{
    "T": "extracted T stage value",
    "N": "extracted N stage value", 
    "stage": "extracted overall stage value",
    "reason": "extracted reasoning or explanation"
}}

Important rules:
- Extract ONLY the stage values (e.g., "T4a", "N2a", "Stage IVa" or "IVa")
- Remove any prefixes like "Stage:" or "T:" from the values
- Remove (M0) from stage if present
- If a field is not found, use empty string ""
- Return ONLY valid JSON, no other text or markdown formatting"""

    messages = [ChatMessage.from_user(extraction_prompt)]
    
    try:
        extraction_response = generator_instance.run(messages=messages)
        extracted_text = extraction_response['replies'][0].text
        
        # Remove code fences if present
        extracted_text = re.sub(r"```(?:json)?\s*([\s\S]*?)```", lambda m: m.group(1), extracted_text).strip()
        
        # Find JSON object
        json_match = re.search(r'\{[\s\S]*?\}', extracted_text)
        if json_match:
            result = json.loads(json_match.group(0))
            
            # Validate and normalize keys
            validated = {
                'T': str(result.get('T', '')),
                'N': str(result.get('N', '')),
                'stage': str(result.get('stage', '')),
                'reason': str(result.get('reason', ''))
            }
            
            return validated
        else:
            print("⚠ LLM extraction did not return valid JSON")
            return None
            
    except Exception as e:
        print(f"⚠ LLM extraction failed: {e}")
        return None

### Evaluation

In [48]:
#write function to get numerix
import re
def extract_number(text: str) :
    """
    Extract first number from text (e.g., 'T4a' → 4, 'IVb' → 4, 'N2b' → 2)
    
    Args:
        text: Input string
    
    Returns:
        First integer found, or None if no number exists
    """
    if not text or not isinstance(text, str):
        return None
    
    match = re.search(r'\d+', text)
    return str(int(match.group())) if match else None


In [49]:
def convert_stage_to_numeric(stage: str):
    """
    Convert stage string to numeric value.
    E.g., 'Stage IV' → 4, 'IVa' → 4, 'IVb' → 4, 'III' → 3
    
    Args:
        stage: Stage string (e.g., 'IV', 'Stage IVa', 'IVb')
    
    Returns:
        Numeric stage (1-4) or None if conversion fails
    """
    if not stage or not isinstance(stage, str):
        return None
    
    stage_upper = stage.upper().strip()
    
    # Roman numeral mapping - check longer numerals FIRST
    roman_map = [
        ('IV', '4'), ('III', '3'), ('II', '2'), ('I', '1')
    ]
    
    # Try to extract roman numeral (longest first to avoid partial matches)
    for roman, num in roman_map:
        if roman in stage_upper:
            return num
    
    # Fallback: extract first digit
    match = re.search(r'\d+', stage_upper)
    return str((match.group())) if match else None

In [50]:
df_sample['T_prediction2']=df_sample['T_prediction'].apply(extract_number)
df_sample['N_prediction2']=df_sample['N_prediction'].apply(extract_number)
df_sample['Stage_prediction2']=df_sample['Stage_prediction'].apply(convert_stage_to_numeric)

In [51]:
df_sample['T_prediction2'].fillna('9',inplace=True)
df_sample['N_prediction2'].fillna('9',inplace=True)
df_sample['Stage_prediction2'].fillna('9',inplace=True)

In [53]:
df_sample['T_stage']=df_sample['T_stage'].astype(str)
df_sample['N_stage']=df_sample['N_stage'].astype(str)
df_sample['Overall_stage']=df_sample['Overall_stage'].astype(str)
df_sample['Stage_prediction2']=df_sample['Stage_prediction2'].astype(str)

In [54]:
#model evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
#evaluate multiclassification
def evaluate_classification(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision_wt = precision_score(y_true, y_pred, average='weighted')
    precision_micro = precision_score(y_true, y_pred, average='micro')
    precision_macro = precision_score(y_true, y_pred, average='macro')

    recall_wt = recall_score(y_true, y_pred, average='weighted')
    recall_micro = recall_score(y_true, y_pred, average='micro')
    recall_macro = recall_score(y_true, y_pred, average='macro')

    f1_wt = f1_score(y_true, y_pred, average='weighted')
    f1_micro = f1_score(y_true, y_pred, average='micro')
    f1_macro = f1_score(y_true, y_pred, average='macro')
    #round all score
    n= 4
    accuracy = round(accuracy,n)
    precision_wt = round(precision_wt,n)
    recall_wt = round(recall_wt,n)
    f1_wt = round(f1_wt,n)
    precision_micro = round(precision_micro,n)
    recall_micro = round(recall_micro,n)
    f1_micro = round(f1_micro,n)
    precision_macro = round(precision_macro,n)
    recall_macro = round(recall_macro,n)
    f1_macro = round(f1_macro,n)

    return accuracy, precision_wt, recall_wt, f1_wt, precision_micro, recall_micro, f1_micro, precision_macro, recall_macro, f1_macro

T stage accuracy

In [55]:
accuracy, precision_wt, recall_wt, f1_wt, precision_micro, recall_micro, f1_micro, precision_macro, recall_macro, f1_macro = evaluate_classification(df_sample['T_stage'], df_sample['T_prediction2'])
accuracy

0.6

N stage accuacy

In [56]:
accuracy, precision_wt, recall_wt, f1_wt, precision_micro, recall_micro, f1_micro, precision_macro, recall_macro, f1_macro = evaluate_classification(df_sample['N_stage'], df_sample['N_prediction2'])
accuracy

0.4

stage 

In [57]:
accuracy, precision_wt, recall_wt, f1_wt, precision_micro, recall_micro, f1_micro, precision_macro, recall_macro, f1_macro = evaluate_classification(df_sample['Overall_stage'], df_sample['Stage_prediction2'])
accuracy

0.6